<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a>一本书的补充代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第五章练习题答案（Chapter 5 Exercise solutions）

In [1]:
from importlib.metadata import PackageNotFoundError, version

def get_package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        # Windows 安装的是 tensorflow-cpu，但导入时仍使用 tensorflow。
        if package_name == "tensorflow":
            return version("tensorflow-cpu")
        raise

pkgs = ["numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # 对于 OpenAI 的预训练权重
       ]
for p in pkgs:
    print(f"{p} version: {get_package_version(p)}")

numpy version: 1.26.4
tiktoken version: 0.13.0
torch version: 2.7.1+cu118
tensorflow version: 2.21.0


 
## 练习 5.1：温度缩放的 softmax 分数和采样概率（Exercise 5.1: Temperature-scaled softmax scores and sampling probabilities）

- 我们可以使用我们在本节中定义的 `print_sampled_tokens` 函数打印“pizza”进行被采样的数量
- 让我们从 5.3.1 节中定义的代码开始

- 如果温度为 0 或 0.1，则采样 0x；如果温度放大到 5，则采样 32x。估计概率为 32/1000 * 100% = 3.2%

- 实际概率为4.3%，包含在重新缩放的softmax概率张量中(`scaled_probas[2][6]`)

- 下面是一个使用第 5 章中的代码的独立示例：

In [2]:
import torch

vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 
inverse_vocab = {v: k for k, v in vocab.items()}

next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

def print_sampled_tokens(probas):
    torch.manual_seed(123)
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    sampled_ids = torch.bincount(torch.tensor(sample))
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")


def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)


temperatures = [1, 0.1, 5]  # 原始温度、较高温度和较低温度
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

- 现在，我们可以迭代 `scaled_probas` 并降低封装情况下的采样频率：

In [4]:
for i, probas in enumerate(scaled_probas):
    print("\n\nTemperature:", temperatures[i])
    print_sampled_tokens(probas)



Temperature: 1
73 x closer
0 x every
0 x effort
582 x forward
2 x inches
0 x moves
0 x pizza
343 x toward


Temperature: 0.1
0 x closer
0 x every
0 x effort
985 x forward
0 x inches
0 x moves
0 x pizza
15 x toward


Temperature: 5
165 x closer
75 x every
42 x effort
239 x forward
71 x inches
46 x moves
32 x pizza
227 x toward
103 x you


- 请注意，当单词对“pizza”进行采样时，采样提供了实际概率的近似值
- 例如，如果采样 32/1000 次，则估计概率为 3.2%
- 要获得实际概率，我们可以直接访问`scaled_probas`中的相应边境来检查概率

- 由于“pizza”是词汇表中的第7个入境，对于温度5，我们按如下方式获得：

In [4]:
temp5_idx = 2
pizza_idx = 6

scaled_probas[temp5_idx][pizza_idx]

tensor(0.0430)

如果温度设置为5，则有4.3%的概率对单词“pizza”进行采样

 
## 练习 5.2：不同的温度和 top-k 设置（Exercise 5.2: Different temperature and top-k settings）

- 温度和top-k的设置都必须根据个人LLM进行调整（一个试错过程，直到产生需要的输出）
-然而，理想的结果也特定于应用程序的
  - 较低的 top-k 和温度会导致少量的随机结果，这在创建教育内容、技术写作或问答、数据分析、代码生成等时是理想的
  - 排名第一的top-k和温度会导致更加合理的调整和随机的输出，这对于头痛风暴任务、创意写作等来说是更理想的

 
## 练习 5.3：解码函数中的确定性行为（Exercise 5.3: Deterministic behavior in the decoding functions）

有多种方法可以使用 `generate` 函数强制确定性行为：

1、设置为`temperature=0.0`；
2.设置`top_k=1`。

下面是一个使用第 5 章中的代码的独立示例：

In [5]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,  # 词汇量
    "context_length": 256,       # 缩短上下文长度（原：1024）
    "emb_dim": 768,       # 嵌入尺寸
    "n_heads": 12,        # 注意力头数量
    "n_layers": 12,       # 层数
    "drop_rate": 0.1,     # dropout 比率
    "qkv_bias": False     # QKV 偏置
}


torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(torch.load("model.pth", weights_only=True))
model.eval();

In [6]:
from gpt_generate import generate, text_to_token_ids, token_ids_to_text
from previous_chapters import generate_text_simple

In [7]:
# 使用 torch.argmax 的确定性函数

start_context = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"




In [8]:
# 确定性行为：无top_k，无温度缩放

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=None,
    temperature=0.0
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"




- 请注意，重新执行之前的代码单元将生成完全相同的生成文本：

In [9]:
# 确定性行为：无top_k，无温度缩放

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=None,
    temperature=0.0
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"




 
## 练习 5.4：继续预训练（Exercise 5.4: Continued pretraining）

- 如果我们仍然在第 5 章中第一次训练模型的 Python 会话中，要继续再训练一个 epoch，我们只需加载我们在主章节中保存的模型和优化器，然后调用 `train_model_simple` 函数

- 需要执行几个步骤才能在新的代码环境中重置此内容
-首先，我们加载分词器、模型和优化器：

In [10]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 词汇量
    "context_length": 256, # 缩短上下文长度（原：1024）
    "emb_dim": 768,        # 嵌入尺寸
    "n_heads": 12,         # 注意力头数量
    "n_layers": 12,        # 层数
    "drop_rate": 0.1,      # dropout 比率
    "qkv_bias": False      # QKV 偏置
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = tiktoken.get_encoding("gpt2")

checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

- 接下来，我们初始化数据加载器：

In [11]:
import os
import requests
from previous_chapters import create_dataloader_v1


file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

# 本书最初使用了下面的代码
# 但是，urllib 使用较旧的协议设置
# 可能会给一些使用 VPN 的读者带来问题。
# 上面的`requests`版本更加健壮
# 在这方面。

"""
import urllib.request

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
"""


# 训练/验证比率
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

-最后，我们使用 `train_model_simple` 函数来训练模型：

In [12]:
from gpt_train import train_model_simple

num_epochs = 1
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)

Ep 1 (Step 000000): Train loss 0.271, Val loss 6.545
Ep 1 (Step 000005): Train loss 0.244, Val loss 6.614
Every effort moves you?"  "Yes--quite insensible to the irony. She wanted him vindicated--and by me!"  He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I


 
## 练习 5.5：预训练模型的训练和验证集损失（Exercise 5.5: Training and validation set losses of the pretrained model）

- 我们可以使用以下代码来计算GPT模型的训练和验证集损失：

```python
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)
```

- 124M参数造成的损失如下：

```
Training loss: 3.754748503367106
Validation loss: 3.559617757797241
```

- 主要观察结果是训练集和验证集的性能具有相同水平
- 这可以有多种解释：

1. OpenAI 训练 GPT-2 训练时，Verdict 不是预数据集的一部分。因此，该模型明显过度训练集，并且在 Verdict 的训练集和验证集部分上表现也同样出色。（验证集损失略低于训练集损失，这在深度学习中并不常见。但是，这可能是由于随机噪声造成的，因为数据集相对较差。实际上，如果没有过度性能，训练集和验证集的预期将大致相同）。

2. 结论是 GPT -2 训练数据集的一部分。在这种情况下，我们无法判断模型是否过度扩增训练数据，验证集也用于训练。为了评估过度扩增的程度，我们需要在 OpenAI 完成 GPT-2 训练后生成一个新的数据集，确保它不会成为预训练的一部分。

下面的代码是这个新笔记本的可重现的独立示例。

In [13]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 词汇量
    "context_length": 256, # 缩短上下文长度（原：1024）
    "emb_dim": 768,        # 嵌入尺寸
    "n_heads": 12,         # 注意力头数量
    "n_layers": 12,        # 层数
    "drop_rate": 0.1,      # dropout 比率
    "qkv_bias": False      # QKV 偏置
}


torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")

In [14]:
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe


In [15]:
# 在字典中定义模型配置以实现紧凑性
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# 复制基本配置并使用特定模型设置进行更新
model_name = "gpt2-small (124M)"  # 型号名称示例
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [16]:
from gpt_generate import load_weights_into_gpt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_weights_into_gpt(gpt, params)
gpt.to(device);

In [17]:
import os
import urllib.request
from previous_chapters import create_dataloader_v1


file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


# 训练/验证比率
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [18]:
from gpt_train import calc_loss_loader

torch.manual_seed(123) # 由于数据加载器中的改组而导致的可重复性
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 3.7547486888037787
Validation loss: 3.5596182346343994


我们还可以对最大的 GPT-2 模型重复此操作，但不要忘记更新上下文长度：

In [19]:
settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")

model_name = "gpt2-xl (1558M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

load_weights_into_gpt(gpt, params)
gpt.to(device)

torch.manual_seed(123)
train_loss = calc_loss_loader(train_loader, gpt, device)
val_loss = calc_loss_loader(val_loader, gpt, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

checkpoint: 100%|███████████████████████████| 77.0/77.0 [00:00<00:00, 43.5kiB/s]
encoder.json: 100%|███████████████████████| 1.04M/1.04M [00:00<00:00, 2.75MiB/s]
hparams.json: 100%|█████████████████████████| 91.0/91.0 [00:00<00:00, 60.2kiB/s]
model.ckpt.data-00000-of-00001: 100%|█████| 6.23G/6.23G [06:02<00:00, 17.2MiB/s]
model.ckpt.index: 100%|████████████████████| 20.7k/20.7k [00:00<00:00, 171kiB/s]
model.ckpt.meta: 100%|████████████████████| 1.84M/1.84M [00:00<00:00, 4.27MiB/s]
vocab.bpe: 100%|████████████████████████████| 456k/456k [00:00<00:00, 1.73MiB/s]


Training loss: 3.3046312861972384
Validation loss: 3.1195147037506104


 
## 练习 5.6：尝试更大的模型（Exercise 5.6: Trying larger models）

- 在本章主代码中，我们实验了最小的 GPT-2模型，它只有124M参数
- 原因是为了降低资源需求
-但是，您可以通过最少的代码更改轻松尝试更大的模型
- 例如，不是在第 5 章中加载 1558M 而不是 124M 模型，我们只需更改以下 2 行代码：

```python
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")
model_name = "gpt2-small (124M)"
```

- 更新后的代码


```python
settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")
model_name = "gpt2-xl (1558M)"
```

In [20]:
import tiktoken
import torch
from previous_chapters import GPTModel


GPT_CONFIG_124M = {
    "vocab_size": 50257,   # 词汇量
    "context_length": 256, # 缩短上下文长度（原：1024）
    "emb_dim": 768,        # 嵌入尺寸
    "n_heads": 12,         # 注意力头数量
    "n_layers": 12,        # 层数
    "drop_rate": 0.1,      # dropout 比率
    "qkv_bias": False      # QKV 偏置
}


tokenizer = tiktoken.get_encoding("gpt2")

In [21]:
from gpt_download import download_and_load_gpt2
from gpt_generate import load_weights_into_gpt


model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

model_name = "gpt2-xl (1558M)"
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

settings, params = download_and_load_gpt2(model_size="1558M", models_dir="gpt2")
load_weights_into_gpt(gpt, params)

File already exists and is up-to-date: gpt2/1558M/checkpoint
File already exists and is up-to-date: gpt2/1558M/encoder.json
File already exists and is up-to-date: gpt2/1558M/hparams.json
File already exists and is up-to-date: gpt2/1558M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/1558M/model.ckpt.index
File already exists and is up-to-date: gpt2/1558M/model.ckpt.meta
File already exists and is up-to-date: gpt2/1558M/vocab.bpe


In [22]:
from gpt_generate import generate, text_to_token_ids, token_ids_to_text

In [23]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you toward finding an ideal life. You don't have to accept your current one at once, because if you do you'll never
